In [1]:
import pandas as pd
import numpy as np


In [5]:
from pathlib import Path

nome_arquivo = '03.BaseDPEvolucaoMensalCisp.csv'
candidatos = [Path(nome_arquivo)]

for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    candidatos.append(base / nome_arquivo)
    candidatos.append(base / 'codigos_py' / 'Analise' / nome_arquivo)

arquivo_csv = next((caminho for caminho in candidatos if caminho.exists()), None)

if arquivo_csv is None:
    raise FileNotFoundError(f'Arquivo não encontrado: {nome_arquivo}')

df = pd.read_csv(arquivo_csv, sep=';', encoding='iso-8859-1')

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
# Se desejasse filtrar anteriormente as colunas poderia fazer assim
# df = 
df = df.dropna(subset=['cisp', 'regiao', 'ano', 'roubo_veiculo'])
# Não serão feitos agrupamentos, uma vez que pode-se fazê-los no Power BI
# No entanto, outliers não são fáceis de calcular no Power BI
# Assim como calcular quartis
q1 = df['roubo_veiculo'].quantile(0.25)
q3 = df['roubo_veiculo'].quantile(0.75)

iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

df_outliers_sup = df[df['roubo_veiculo'] > lim_sup].copy()
df_outliers_inf = df[df['roubo_veiculo'] < lim_inf].copy()
# Esse .copy tem a função de evitar que esses dataframes sejam linkados

df_outliers_sup['flag'] = 'outlier_sup'
df_outliers_inf['flag'] = 'outlier_inf'
# Vamos unir os dois outliers para enviá-los para o Power BI
resultado_outlier = pd.concat([df_outliers_inf, df_outliers_sup])
resultado_outlier.to_csv('dp_outliers.csv', sep=';', encoding='utf-8-sig', index=False)
resultado_outlier.head()

,cisp,mes,ano,mes_ano,aisp,risp,munic,mcirc,regiao,hom_doloso,...,cmba,ameaca,pessoas_desaparecidas,encontro_cadaver,encontro_ossada,pol_militares_mortos_serv,pol_civis_mortos_serv,registro_ocorrencias,fase,flag
14,19,1,2003,2003m01,6,1,Rio de Janeiro,3304557,Capital,0,...,NaN,52,1,2,0,0,0,569,3,outlier_sup
16,21,1,2003,2003m01,22,1,Rio de Janeiro,3304557,Capital,23,...,NaN,53,8,5,0,1,1,991,3,outlier_sup
17,22,1,2003,2003m01,16,1,Rio de Janeiro,3304557,Capital,2,...,NaN,43,4,3,0,0,0,687,3,outlier_sup
18,23,1,2003,2003m01,3,1,Rio de Janeiro,3304557,Capital,1,...,NaN,56,0,0,0,0,0,790,3,outlier_sup
19,24,1,2003,2003m01,3,1,Rio de Janeiro,3304557,Capital,3,...,NaN,55,6,1,0,0,0,502,3,outlier_sup
